In [5]:
import numpy as np
import pandas as pd
from math import radians, cos, sin, sqrt, atan2

def calcular_distancia_haversine(lat1, lon1, lat2, lon2):
    """Calcula a distância entre dois pontos geográficos (em km) usando a fórmula de Haversine."""
    R = 6371  # Raio médio da Terra em km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

def calcular_matriz_distancias(df):
    """Gera matriz de distâncias entre todos os HUBs do DataFrame."""
    n = len(df)
    matriz = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            dist = calcular_distancia_haversine(
                df.iloc[i]['Latitude'], df.iloc[i]['Longitude'],
                df.iloc[j]['Latitude'], df.iloc[j]['Longitude']
            )
            matriz[i, j] = dist
            matriz[j, i] = dist
    return matriz

def analisar_hubs(df, regiao=None, uf=None, distancia_minima_km=50):
    """
    Analisa distribuição de HUBs e identifica proximidades geográficas.

    Parâmetros:
        df: DataFrame com colunas ['HUB', 'REGIÃO', 'UF', 'CIDADE', 'Latitude', 'Longitude']
        regiao: (opcional) nome da região para filtrar
        uf: (opcional) UF para filtrar
        distancia_minima_km: distância mínima considerada “muito próxima”

    Retorna:
        dicionário com estatísticas, hubs próximos e recomendações
    """
    
    # 1️⃣ Limpeza inicial
    df_clean = df.dropna(subset=['Latitude', 'Longitude']).copy()

    # 2️⃣ Aplicar filtros opcionais
    if regiao:
        df_clean = df_clean[df_clean['REGIÃO'] == regiao]
    if uf:
        df_clean = df_clean[df_clean['UF'] == uf]

    if len(df_clean) == 0:
        return {'erro': 'Nenhum HUB encontrado com os filtros aplicados'}

    # 3️⃣ Calcular matriz de distâncias entre os hubs
    matriz_dist = calcular_matriz_distancias(df_clean)

    # 4️⃣ Identificar hubs muito próximos
    hubs_proximos = []
    df_reset = df_clean.reset_index(drop=True)
    for i in range(len(df_reset)):
        for j in range(i + 1, len(df_reset)):
            dist = matriz_dist[i, j]
            if dist < distancia_minima_km:
                hubs_proximos.append({
                    'hub1': df_reset.loc[i, 'HUB'],
                    'hub2': df_reset.loc[j, 'HUB'],
                    'distancia_km': round(dist, 2),
                    'cidade1': df_reset.loc[i, 'CIDADE'],
                    'cidade2': df_reset.loc[j, 'CIDADE']
                })

    # 5️⃣ Calcular centro geométrico
    centro_lat = df_clean['Latitude'].mean()
    centro_lon = df_clean['Longitude'].mean()

    # 6️⃣ Calcular dispersão em torno do centro
    distancias_centro = [
        calcular_distancia_haversine(centro_lat, centro_lon, row['Latitude'], row['Longitude'])
        for _, row in df_clean.iterrows()
    ]

    # 7️⃣ Montar resultado
    return {
        'resumo': {
            'total_hubs': len(df_clean),
            'regiao': regiao,
            'uf': uf,
            'pares_proximos': len(hubs_proximos)
        },
        'centro_geometrico': {
            'latitude': centro_lat,
            'longitude': centro_lon
        },
        'hubs_muito_proximos': hubs_proximos,
        'estatisticas': {
            'distancia_media_centro_km': round(np.mean(distancias_centro), 2),
            'distancia_maxima_centro_km': round(np.max(distancias_centro), 2),
            'distancia_minima_centro_km': round(np.min(distancias_centro), 2),
            'dispersao_std_km': round(np.std(distancias_centro), 2)
        },
        'recomendacoes': [
            f"Existem {len(hubs_proximos)} pares de HUBs muito próximos (< {distancia_minima_km} km)",
            ("Considere consolidar ou realocar HUBs próximos" 
                if len(hubs_proximos) > 0 
                else "Distribuição adequada de HUBs")
        ]
    }

In [7]:
import pandas as pd

dados = [
    ["HUB SP1", "Sudeste", "SP", "São Paulo", -23.5505, -46.6333],
    ["HUB RJ1", "Sudeste", "RJ", "Rio de Janeiro", -22.9068, -43.1729],
    ["HUB POA1", "Sul", "RS", "Porto Alegre", -30.033, -51.23],
    ["HUB CWB1", "Sul", "PR", "Curitiba", -25.4284, -49.2733],
]

df = pd.DataFrame(dados, columns=["HUB","REGIÃO","UF","CIDADE","Latitude","Longitude"])
print(df)


        HUB   REGIÃO  UF          CIDADE  Latitude  Longitude
0   HUB SP1  Sudeste  SP       São Paulo  -23.5505   -46.6333
1   HUB RJ1  Sudeste  RJ  Rio de Janeiro  -22.9068   -43.1729
2  HUB POA1      Sul  RS    Porto Alegre  -30.0330   -51.2300
3  HUB CWB1      Sul  PR        Curitiba  -25.4284   -49.2733


In [8]:
resultado = analisar_hubs(df, distancia_minima_km=400)
print(resultado)

{'resumo': {'total_hubs': 4, 'regiao': None, 'uf': None, 'pares_proximos': 2}, 'centro_geometrico': {'latitude': np.float64(-25.479675), 'longitude': np.float64(-47.577374999999996)}, 'hubs_muito_proximos': [{'hub1': 'HUB SP1', 'hub2': 'HUB RJ1', 'distancia_km': np.float64(360.75), 'cidade1': 'São Paulo', 'cidade2': 'Rio de Janeiro'}, {'hub1': 'HUB SP1', 'hub2': 'HUB CWB1', 'distancia_km': np.float64(339.05), 'cidade1': 'São Paulo', 'cidade2': 'Curitiba'}], 'estatisticas': {'distancia_media_centro_km': np.float64(389.1), 'distancia_maxima_centro_km': np.float64(620.8), 'distancia_minima_centro_km': np.float64(170.37), 'dispersao_std_km': np.float64(190.6)}, 'recomendacoes': ['Existem 2 pares de HUBs muito próximos (< 400 km)', 'Considere consolidar ou realocar HUBs próximos']}


In [9]:
{
 'resumo': {...},
 'centro_geometrico': {...},
 'hubs_muito_proximos': [...],
 'estatisticas': {...},
 'recomendacoes': [...]
}

{'resumo': {Ellipsis},
 'centro_geometrico': {Ellipsis},
 'hubs_muito_proximos': [Ellipsis],
 'estatisticas': {Ellipsis},
 'recomendacoes': [Ellipsis]}

In [10]:
resultado_sul = analisar_hubs(df, regiao="Sul", distancia_minima_km=300)
print(resultado_sul)

{'resumo': {'total_hubs': 2, 'regiao': 'Sul', 'uf': None, 'pares_proximos': 0}, 'centro_geometrico': {'latitude': np.float64(-27.7307), 'longitude': np.float64(-50.25165)}, 'hubs_muito_proximos': [], 'estatisticas': {'distancia_media_centro_km': np.float64(273.51), 'distancia_maxima_centro_km': np.float64(273.86), 'distancia_minima_centro_km': np.float64(273.15), 'dispersao_std_km': np.float64(0.36)}, 'recomendacoes': ['Existem 0 pares de HUBs muito próximos (< 300 km)', 'Distribuição adequada de HUBs']}


In [11]:
print("Resumo:")
print(resultado['resumo'])

print("\nCentro geométrico:")
print(resultado['centro_geometrico'])

print("\nHUBs muito próximos:")
print(resultado['hubs_muito_proximos'])

print("\nEstatísticas:")
print(resultado['estatisticas'])

print("\nRecomendações:")
print(resultado['recomendacoes'])

Resumo:
{'total_hubs': 4, 'regiao': None, 'uf': None, 'pares_proximos': 2}

Centro geométrico:
{'latitude': np.float64(-25.479675), 'longitude': np.float64(-47.577374999999996)}

HUBs muito próximos:
[{'hub1': 'HUB SP1', 'hub2': 'HUB RJ1', 'distancia_km': np.float64(360.75), 'cidade1': 'São Paulo', 'cidade2': 'Rio de Janeiro'}, {'hub1': 'HUB SP1', 'hub2': 'HUB CWB1', 'distancia_km': np.float64(339.05), 'cidade1': 'São Paulo', 'cidade2': 'Curitiba'}]

Estatísticas:
{'distancia_media_centro_km': np.float64(389.1), 'distancia_maxima_centro_km': np.float64(620.8), 'distancia_minima_centro_km': np.float64(170.37), 'dispersao_std_km': np.float64(190.6)}

Recomendações:
['Existem 2 pares de HUBs muito próximos (< 400 km)', 'Considere consolidar ou realocar HUBs próximos']


In [ ]:
import folium

# Pega localização do centro
lat_cent = resultado['centro_geometrico']['latitude']
lon_cent = resultado['centro_geometrico']['longitude']

# Cria o mapa
mapa = folium.Map(location=[lat_cent, lon_cent], zoom_start=5)

# Adiciona marcadores dos HUBs
for _, row in df.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=row['HUB']
    ).add_to(mapa)

# Adiciona o centro geométrico
folium.Marker(
    location=[lat_cent, lon_cent],
    popup="Centro Geométrico (média)",
    icon=folium.Icon(color="red")

).add_to(mapa)

mapa
